### ЗАДАЧА: Архитектура компенсаций (ABC + наследование + композиция)

В компании есть сотрудники разных ролей: разработчики, менеджеры и аналитики.
Нужно собрать ООП-модель расчёта выплат и отчёт по отделам.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Абстрактный класс `EmployeeBase`:
   - поля `name`, `base_salary`, `department`
   - абстрактный метод `total_pay()`
   - метод `short()` для короткого описания сотрудника.

2. Классы-наследники:
   - `Developer(name, base_salary, department, overtime_hours, overtime_rate)`
   - `Manager(name, base_salary, department, bonus)`
   - `Analyst(name, base_salary, department, reports_done, report_rate)`
   - у каждого свой расчёт `total_pay()`.

3. Класс `DepartmentBudget` (композиция):
   - хранит `department_name` и список сотрудников отдела
   - методы `add_employee(emp)`, `department_total()`, `top_paid()`.

4. Класс `PayrollService`:
   - принимает список сотрудников
   - метод `total_company_payroll()`
   - метод `totals_by_department()` -> словарь `{department: сумма}`
   - метод `highest_paid_employee()`.

5. Добавить dunder-методы:
   - `__repr__` для сотрудников
   - `__lt__` для сравнения сотрудников по `total_pay()`.

6. На основе входных данных создать объекты и вывести:
   - общий payroll,
   - выплаты по отделам,
   - самого дорогого сотрудника.

ПОДСКАЗКИ ПО РАСЧЁТАМ:
- `Developer.total_pay()` -> `base_salary + overtime_hours * overtime_rate`.
- `Manager.total_pay()` -> `base_salary + bonus`.
- `Analyst.total_pay()` -> `base_salary + reports_done * report_rate`.
- В `totals_by_department()` удобно использовать `dict.get(dep, 0)`.
- В `top_paid()` и `highest_paid_employee()` использовать `max(..., key=...)`.


In [1]:
from abc import ABC, abstractmethod

staff_rows = [
    ("Developer", "Алиса", 2200, "Engineering", 12, 25),
    ("Manager", "Боб", 2800, "Engineering", 900),
    ("Analyst", "Кира", 2000, "Data", 14, 40),
    ("Developer", "Данил", 2400, "Engineering", 4, 25),
    ("Manager", "Ева", 2600, "Data", 600),
    ("Analyst", "Жан", 1900, "Operations", 20, 30),
]


class EmployeeBase(ABC):
    def __init__(self, name, base_salary, department):
        self.name = name
        self.base_salary = base_salary
        self.department = department
    #  __init__(name, base_salary, department)

    @abstractmethod
    def short(self) -> str:
        pass
    #  short() -> строка "Имя (Department)"

    @abstractmethod
    def total_pay(self) -> float :
        pass
    
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({self.name}, {self.total_pay()})"
    # __repr__

    def __lt__(self, other) -> bool:
        if not isinstance(other, EmployeeBase):
            return NotImplemented
        return self.total_pay() < other.total_pay()
    #  __lt__(other) -> сравнение по total_pay()


class Developer(EmployeeBase):
    def __init__(self, name, base_salary, department, overtime_hours, overtime_rate):
        super().__init__(name, base_salary, department)
        self.overtime_hours = overtime_hours
        self.overtime_rate = overtime_rate
    # TODO: __init__(name, base_salary, department, overtime_hours, overtime_rate)

    def total_pay(self):
        return self.base_salary + self.overtime_hours * self.overtime_rate
    #  total_pay()

    def short(self) -> str:
        return f"{self.name} ({self.department})"


class Manager(EmployeeBase):
    def __init__(self, name, base_salary, department, bonus):
        super().__init__(name, base_salary, department)
        self.bonus = bonus
    # TODO: __init__(name, base_salary, department, bonus)

    def total_pay(self):
        return self.base_salary + self.bonus
    # TODO: total_pay()
    
    def short(self) -> str:
        return f"{self.name} ({self.department})"

class Analyst(EmployeeBase):
    def __init__(self, name, base_salary, department, reports_done, report_rate):
        super().__init__(name, base_salary, department)
        self.reports_done = reports_done
        self. report_rate = report_rate
    # TODO: __init__(name, base_salary, department, reports_done, report_rate)

    def total_pay(self):
        return self.base_salary + self.reports_done * self.report_rate
    # TODO: total_pay()

    def short(self) -> str:
        return f"{self.name} ({self.department})"

class DepartmentBudget:
    def __init__(self, department_name):
        self.department_name = department_name
    # TODO: __init__(department_name)
        self.employees = []

    def add_employee(self, emp: EmployeeBase):
        if emp.department == self.department_name:
            self.employees.append(emp)
    # TODO: add_employee(emp)

    def department_total(self) -> float:
        return sum(emp.total_pay() for emp in self.employees)
    # TODO: department_total()


    # TODO: top_paid()
    def top_paid(self) -> EmployeeBase:
        if not self.employees:
            return None
        return max(self.employees, key=lambda emp: emp.total_pay())


class PayrollService:
    def __init__(self, employees):
        self.employees = employees
    # TODO: __init__(employees)
        self.departments = {}
        for emp in employees:
            if emp.department not in self.departments:
                self.departments[emp.department] = DepartmentBudget(emp.department)
            self.departments[emp.department].add_employee(emp)

    # TODO: total_company_payroll()
    def total_company_payroll(self) -> float:
        return sum(emp.total_pay() for emp in self.employees)

    # TODO: totals_by_department()
    def totals_by_department(self) -> dict[str, float]:
        return {dept: budget.department_total() for dept, budget in self.departments.items()}

    # TODO: highest_paid_employee()
    def highest_paid_employee(self) -> EmployeeBase:
        if not self.employees:
            return None
        return max(self.employees, key=lambda emp: emp.total_pay())

employees = []
for row in staff_rows:
    position = row[0]
    if position == "Developer":
        emp = Developer(row[1], row[2], row[3], row[4], row[5])
    elif position == "Manager":
        emp = Manager(row[1], row[2], row[3], row[4])
    elif position == "Analyst":
        emp = Analyst(row[1], row[2], row[3], row[4], row[5])
    else:
        raise ValueError(f"Неизвестная позиция: {position}")
    employees.append(emp)
# TODO: создать объекты сотрудников из staff_rows

payroll = PayrollService(employees)
print("Общий доход компании:", payroll.total_company_payroll())
# TODO: создать PayrollService и вывести общий payroll

print("\nСуммы по отделам:")
for dept, total in payroll.totals_by_department().items():
    print(f"  {dept}: {total}")
# TODO: вывести суммы по отделам

highest_paid = payroll.highest_paid_employee()
print(f"\nСамый высокооплачиваемый сотрудник: {highest_paid.short()} с зарплатой {highest_paid.total_pay()}")
# TODO: вывести самого высокооплачиваемого сотрудника


Общий доход компании: 16960

Суммы по отделам:
  Engineering: 8700
  Data: 5760
  Operations: 2500

Самый высокооплачиваемый сотрудник: Боб (Engineering) с зарплатой 3700
